In [3]:
%matplotlib inline
# Pandas is a nice utilitiy that enables some easy data manipulation, especially from a csv
import pandas as pd
# Numpy lets us work with arrays
import numpy as np
import random
# Sklearn provides various modules with a common API
from sklearn import svm, tree, neighbors, neural_network, ensemble
from sklearn.model_selection import train_test_split
# For optimizing our models' hyperparameters
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# Set random seed
random.seed(100)
np.random.seed(100)

In [4]:
# Read in our csv files downloaded from Kaggle
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

In [5]:
random.seed(100)
np.random.seed(100)

train_data.drop(columns=['Name', 'Ticket', 'Cabin'], inplace=True)
train_data.set_index(keys=['PassengerId'], drop=True, inplace=True)

test_data.drop(columns=['Name', 'Ticket', 'Cabin'], inplace=True)
test_data.set_index(keys=['PassengerId'], drop=True, inplace=True)

train_nan_map = {'Age': train_data['Age'].mean(), 'Fare': train_data['Fare'].mean(), 'Embarked': train_data['Embarked'].mode()[0]}
test_nan_map = {'Age': test_data['Age'].mean(), 'Fare': test_data['Fare'].mean(), 'Embarked': test_data['Embarked'].mode()[0]}

train_data.fillna(value=train_nan_map, inplace=True)
test_data.fillna(value=test_nan_map, inplace=True)

columns_map = {'Embarked': {'C': 0, 'Q': 1, 'S': 2}, 'Sex': {'male': 0, 'female': 1}}
# Adding below line to surpress deprication warnings
pd.set_option('future.no_silent_downcasting', True)
train_data.replace(columns_map, inplace=True)
test_data.replace(columns_map, inplace=True)

X_train = train_data.loc[:, train_data.columns != 'Survived']
y_train = train_data.loc[:, 'Survived']

X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.33, random_state=10)

/var/folders/ck/k3xlx_417p77nqyjfb7klv6w0000gn/T/ipykernel_52241/3956895788.py:18: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)


Andy - Boosted Tree

In [6]:
# Optimize Hyperparameters
random.seed(100)
np.random.seed(100)

param_grid = {
    'learning_rate': [.01, .02, .04],
    'n_estimators': [50, 60, 70, 80, 90],
    'max_depth': [2,3,4],
    'subsample': [0.9, 1.0, 1.1]
}

boost_clf = ensemble.GradientBoostingClassifier(random_state=100)

grid_search = GridSearchCV(
    estimator=boost_clf,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 135 candidates, totalling 675 fits
Best params: {'learning_rate': 0.04, 'max_depth': 3, 'n_estimators': 80, 'subsample': 1.0}
Best CV score: 0.8651


/opt/anaconda3/envs/emade/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:489: FitFailedWarning: 
225 fits failed out of a total of 675.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
225 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/emade/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/emade/lib/python3.13/site-packages/sklearn/base.py", line 1393, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/anaconda3/envs/emade/lib/python3.13/site-packages/s

In [7]:
optimal_boost_clf = grid_search.best_estimator_
print(optimal_boost_clf.score(X_test, y_test))

y_pred = optimal_boost_clf.predict(X_test)
y_truth = y_test

0.823728813559322


In [8]:
tn, fp, fn, tp = confusion_matrix(y_truth, y_pred).ravel()

print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)



True Negatives: 169
False Positives: 22
False Negatives: 30
True Positives: 74


Stevie - SVM

In [ ]:
from sklearn import svm
from sklearn.metrics import confusion_matrix

param_grid = [{'C': c, 'class_weight': cw} for c in [0.01, 0.1, 1, 10, 100] for cw in [None, 'balanced']]

svm_results = []
for params in param_grid:
    clf = svm.SVC(kernel='linear', **params)
    clf.fit(X_train.values, y_train.values)
    y_pred = clf.predict(X_test.values)
    tn, fp, fn, tp = confusion_matrix(y_test.values, y_pred).ravel()
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    accuracy = clf.score(X_test.values, y_test.values)
    svm_results.append({'Model': f"SVM (C={params['C']}, class_weight={params['class_weight']})",
                         'FPR': fpr, 'FNR': fnr, 'Accuracy': accuracy})

svm_results_df = pd.DataFrame(svm_results)
svm_results_df.to_csv('svm_gridsearch_results.csv', index=False)
svm_results_df

Asher - Logistic Classification

In [ ]:
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"],  # liblinear supports both l1 and l2
    "class_weight": [None, "balanced"],
}
grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring="accuracy")
grid.fit(X_train.values, y_train.values)

print("Best C:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

log_reg = grid.best_estimator_
print("Held-out test accuracy:", log_reg.score(X_test.values, y_test.values))

y_pred = log_reg.predict(X_test.values)
y_truth = y_test.values

Best C: {'C': 0.1, 'class_weight': None, 'penalty': 'l1', 'solver': 'liblinear'}
Best CV accuracy: 0.795252100840336
Held-out test accuracy: 0.7932203389830509


In [ ]:
tn, fp, fn, tp = confusion_matrix(y_truth, y_pred).ravel()
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()


# Compute confusion matrix
cnf_matrix = confusion_matrix(y_truth, y_pred)
class_names = ['0', '1']
np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=class_names,
                      title='Confusion matrix, without normalization')

plt.show()

NameError: name 'plt' is not defined

Shrey - KNN

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid = {
    'knn__n_neighbors': [7, 9, 11, 13, 15, 17, 19, 21, 23, 25],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}

knn_grid = GridSearchCV(
    knn_pipeline,
    param_grid,
    cv=5,
    scoring='roc_auc'
)

knn_grid.fit(X_train.values, y_train.values)

best_knn = knn_grid.best_estimator_

print("Best Parameters:", knn_grid.best_params_)
print("Best CV ROC AUC:", round(knn_grid.best_score_, 4))

In [ ]:
best_knn = knn_grid.best_estimator_

knn_accuracy = best_knn.score(X_test.values, y_test.values)

print("KNN Test Accuracy:", round(knn_accuracy * 100, 2), "%")

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

knn_probabilities = best_knn.predict_proba(X_test.values)[:, 1]

threshold = 0.35
knn_predictions = (knn_probabilities >= threshold).astype(int)

cm = confusion_matrix(y_test.values, knn_predictions)
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test.values, knn_predictions)
precision = precision_score(y_test.values, knn_predictions)
recall = recall_score(y_test.values, knn_predictions)
f1 = f1_score(y_test.values, knn_predictions)
roc_auc = roc_auc_score(y_test.values, knn_probabilities)

print("Best Parameters:", knn_grid.best_params_)
print("Threshold:", threshold)
print("Confusion Matrix:")
print(cm)

print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

print("Accuracy:", round(accuracy * 100, 2), "%")
print("Precision:", round(precision * 100, 2), "%")
print("Recall:", round(recall * 100, 2), "%")
print("F1 Score:", round(f1 * 100, 2), "%")
print("ROC AUC:", round(roc_auc, 4))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Did Not Survive", "Survived"]
)

disp.plot(cmap="Blues", colorbar=False)
plt.title("KNN Confusion Matrix")
plt.show()

In [ ]:
# Use all labeled training data for the final model
X_full = train_data.loc[:, train_data.columns != 'Survived']
y_full = train_data.loc[:, 'Survived']

final_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# Use the parameters GridSearchCV selected
final_knn.set_params(**knn_grid.best_params_)

# Train on all available labeled Titanic data
final_knn.fit(X_full.values, y_full.values)

# Predict probabilities for actual test.csv
final_probabilities = final_knn.predict_proba(test_data.values)[:, 1]

# Same threshold used for your final model
final_predictions = (final_probabilities >= 0.35).astype(int)

submission = pd.DataFrame({
    'PassengerId': test_data.index,
    'Survived': final_predictions
})

submission.to_csv('shrey_knn_submission.csv', index=False)

print(submission.head())
print("Shape:", submission.shape)

Val - Random Forest

In [ ]:
# Random Forest Classifier
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

classifier = RandomForestClassifier(n_estimators=200, random_state=42)
classifier.fit(X_train, y_train)
print(classifier.score(X_test, y_test))
y_pred = classifier.predict(X_test)
y_truth = y_test.values

feature_importances = classifier.feature_importances_

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_truth, y_pred).ravel()
print("Confusion Matrix")
print(confusion_matrix(y_truth, y_pred, labels=[0, 1]))
print("")
print("True Negatives", tn)
print("False Positives", fp)
print("False Negatives", fn)
print("True Positives", tp)

In [ ]:
import itertools
import matplotlib.pyplot as plt

def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()


# Compute confusion matrix
cnf_matrix = confusion_matrix(y_truth, y_pred)
class_names=['0', '1']
np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=class_names,
                      title='Confusion matrix, without normalization')

plt.show()

Plot Pareto